## Module 8: Weight Initialization and Batch Normalization

In [3]:
# Too Large Wegihts catastrophe when using tanh activation: the network gets saturated because 
# of its always equal to ~(-1 or 1)
import torch
import torch.nn as nn
torch.manual_seed(42)
x = torch.randn(1, 100)

for layer in range(5):
    w = torch.randn(100, 100) * 10.0
    z = torch.tanh(x @ w)
print(f'After 5 layers: mean = {z.mean()}, std = {z.std()}')


# Too small weight catastrophe: all weights collapse to near-zero
x = torch.randn(1, 100)
for layer in range(5):
    w = torch.randn(100, 100) * 0.001
    z = torch.tanh(x @ w) 
print(f'After 5 layers: mean = {z.mean()}, std = {z.std()}')

After 5 layers: mean = -0.029589006677269936, std = 0.9990899562835693
After 5 layers: mean = 0.00012704536493401974, std = 0.008628721348941326


In [5]:
# Xavier/Glorot Initialization
import torch
import torch.nn as nn
model = nn.Sequential(nn.Linear(100, 64), nn.Tanh(), nn.Linear(64, 32), nn.Tanh(), nn.Linear(32, 1))

def init_xavier(model):
    for module in model.modules():
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            nn.init.zeros_(module.bias)

init_xavier(model)

x = torch.randn(1000, 100)
with torch.no_grad():
    print(f'Input std: {x.std():.4f}')
    z1 = x @ model[0].weight.T + model[0].bias
    a1 = torch.tanh(z1)
    print(f'After layer 1 std: {a1.std():.4f}')
    z2 = a1 @ model[2].weight.T + model[2].bias
    a2 = torch.tanh(z2)
    print(f'After layer 2 std: {a2.std():.4f}')

Input std: 0.9990
After layer 1 std: 0.6564
After layer 2 std: 0.5343


In [7]:
# He/Kaiming Initialization
def init_he(model):
    for module in model.modules():
        if isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(
                module.weight,
                mode = 'fan_in',
                nonlinearity = 'relu'
            )
            nn.init.zeros_(module.bias)

relu_model = nn.Sequential(nn.Linear(100, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), 
                           nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))
init_he(relu_model)

x = torch.randn(1000, 100)
with torch.no_grad():
    print(f'Input std: {x.std():.4f}')
    z1 = x @ relu_model[0].weight.T + relu_model[0].bias
    a1 = torch.relu(z1)
    print(f'After ReLU 1 std: {a1.std():.4f}')
    z2 = a1 @ relu_model[2].weight.T + relu_model[2].bias
    a2 = torch.relu(z2)
    print(f'After ReLU 2 std: {a2.std():.4f}')
    z3 = a2 @ relu_model[4].weight.T + relu_model[4].bias
    a3 = torch.relu(z3)
    print(f'After ReLU 3 std: {a3.std():.4f}')

Input std: 0.9977
After ReLU 1 std: 0.8243
After ReLU 2 std: 0.8207
After ReLU 3 std: 0.8496


In [11]:
# Comparing with and without he init
def check_variance_propagation(init_std = None, use_he = False, n_layers = 10):
    torch.manual_seed(42)
    x = torch.randn(256, 256)

    print(f"\n{'Layer':>6}  {'Activation Std':>16}  {'Status':>12}")
    print(f"{'Input':>6}  {x.std():>16.4f}  {'OK':>12}")

    for i in range(n_layers):
        w = torch.randn(256, 256)
        if use_he:
            w = w * (2.0 / 256) ** 0.5
        elif init_std:
            w = w * init_std
        x = torch.relu(x @ w)

        status = 'OK' if 0.1 < x.std() < 10 else ('VANISHED' if x.std() < 0.1 else 'EXPLODED')
        print(f'{i + 1:>6}  {x.std():>16.6f}  {status:>12}')

print("=== Too-small init (std=0.01) ===")
check_variance_propagation(init_std=0.01, n_layers=8)

print("\n=== Too-large init (std=1.0) ===")
check_variance_propagation(init_std=1.0, n_layers=8)

print("\n=== He initialization ===")
check_variance_propagation(use_he=True, n_layers=8)

=== Too-small init (std=0.01) ===

 Layer    Activation Std        Status
 Input            1.0060            OK
     1          0.094546      VANISHED
     2          0.010532      VANISHED
     3          0.001173      VANISHED
     4          0.000129      VANISHED
     5          0.000014      VANISHED
     6          0.000002      VANISHED
     7          0.000000      VANISHED
     8          0.000000      VANISHED

=== Too-large init (std=1.0) ===

 Layer    Activation Std        Status
 Input            1.0060            OK
     1          9.454577            OK
     2        105.316086      EXPLODED
     3       1173.171631      EXPLODED
     4      12907.994141      EXPLODED
     5     138377.078125      EXPLODED
     6    1573055.500000      EXPLODED
     7   16890126.000000      EXPLODED
     8  175044784.000000      EXPLODED

=== He initialization ===

 Layer    Activation Std        Status
 Input            1.0060            OK
     1          0.835674            OK
     

In [12]:
# Gradient Clipping: For exploding grads
import torch
import torch.nn as nn
import torch.optim as optim

model     = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 1))
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

X = torch.randn(32, 10)
y = torch.randn(32, 1)

optimizer.zero_grad()
pred = model(X)
loss = criterion(pred, y)
loss.backward()

grad_norm_before = 0.0
for p in model.parameters():
    if p.grad is not None:
        grad_norm_before += p.grad.data.norm(2).item() ** 2
grad_norm_before = grad_norm_before ** 0.5

max_norm = 1.0
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = max_norm)

grad_norm_after = 0.0
for p in model.parameters():
    if p.grad is not None:
        grad_norm_after += p.grad.data.norm(2).item() ** 2
grad_norm_after = grad_norm_after ** 0.5

print(f"Gradient norm before clip: {grad_norm_before:.4f}")
print(f"Gradient norm after clip:  {grad_norm_after:.4f}")
print(f"Max norm threshold:        {max_norm:.4f}")

optimizer.step()

Gradient norm before clip: 1.2716
Gradient norm after clip:  1.0000
Max norm threshold:        1.0000


In [14]:
# Batch Normalization
class DeepNet(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 256), nn.ReLU(), nn.BatchNorm1d(256),      
            nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128),
            nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64),
            nn.Linear(64, num_classes)  # Output layer — NO BatchNorm here
        )

    def forward(self, x):
        return self.net(x)

model = DeepNet(in_features = 100, num_classes = 10)
total_params = sum(p.numel() for p in model.parameters())
bn_params = sum(p.numel() for name, p in model.named_parameters()
               if 'bn' in name or 'batch_norm' in name.lower() or 'weight' in name and 'norm' in name)
print(f'Total parameters: {total_params:,}')

bn_layer = model.net[2]   # first BatchNorm layer
print(f"\nBatchNorm1d(256) state:")
print(f"  weight (γ) shape: {bn_layer.weight.shape}")
print(f"  bias   (β) shape: {bn_layer.bias.shape}")
print(f"  running_mean:     {bn_layer.running_mean.shape}")
print(f"  running_var:      {bn_layer.running_var.shape}")
print(f"  γ initialized to: {bn_layer.weight[0].item():.1f}")
print(f"  β initialized to: {bn_layer.bias[0].item():.1f}")
print(f"  momentum:         {bn_layer.momentum}")
print(f"  epsilon:          {bn_layer.eps}")

Total parameters: 68,554

BatchNorm1d(256) state:
  weight (γ) shape: torch.Size([256])
  bias   (β) shape: torch.Size([256])
  running_mean:     torch.Size([256])
  running_var:      torch.Size([256])
  γ initialized to: 1.0
  β initialized to: 0.0
  momentum:         0.1
  epsilon:          1e-05


In [2]:
# BatchNorm's effect on Activation stats
import torch
import torch.nn as nn

torch.manual_seed(42)

no_bn_model = nn.Sequential(
    nn.Linear(50, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(),
    nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 1))

with_bn_model = nn.Sequential(
    nn.Linear(50, 256), nn.ReLU(), nn.BatchNorm1d(256), 
    nn.Linear(256, 256), nn.ReLU(), nn.BatchNorm1d(256),
    nn.Linear(256, 256), nn.ReLU(), nn.BatchNorm1d(256),
    nn.Linear(256, 256), nn.ReLU(), nn.BatchNorm1d(256),
    nn.Linear(256, 1))

X = torch.randn(64, 50)

print("Activation statistics without BatchNorm:")
no_bn_model.train()
x = X
for i, layer in enumerate(no_bn_model):
    x = layer(x)
    if isinstance(layer, nn.ReLU):
        print(f"  After ReLU {i}: mean={x.mean():.3f}, std={x.std():.3f}")

print("\nActivation statistics WITH BatchNorm:")
with_bn_model.train()
x = X
for i, layer in enumerate(with_bn_model):
    x = layer(x)
    if isinstance(layer, nn.BatchNorm1d):
        print(f"  After BN   {i}: mean={x.mean():.3f}, std={x.std():.3f}")

Activation statistics without BatchNorm:
  After ReLU 1: mean=0.225, std=0.333
  After ReLU 3: mean=0.096, std=0.138
  After ReLU 5: mean=0.041, std=0.059
  After ReLU 7: mean=0.026, std=0.034

Activation statistics WITH BatchNorm:
  After BN   2: mean=-0.000, std=1.000
  After BN   5: mean=-0.000, std=1.000
  After BN   8: mean=-0.000, std=1.000
  After BN   11: mean=0.000, std=1.000


In [3]:
# Layer Normalization
ln = nn.LayerNorm(normalized_shape=256)

x = torch.randn(32, 256)
out = ln(x)
print(f"Mean per sample: {out.mean(dim=1).abs().max():.6f}")   # ≈ 0
print(f"Std per sample:  {out.std(dim=1).mean():.4f}")

Mean per sample: 0.000000
Std per sample:  1.0020


In [7]:
# Comparing Initialization and BatchNorm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Dataset
n_train, n_val = 1000, 300
n_feat = 50

X_train = torch.randn(n_train, n_feat)
X_val   = torch.randn(n_val, n_feat)
y_train = (X_train[:, :5].sum(dim=1) > 0).float().unsqueeze(1)
y_val   = (X_val[:,   :5].sum(dim=1) > 0).float().unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val),     batch_size=128, shuffle=False)

# 2. Model
def build_model(use_batchnorm = False, init_type = 'default'):
    layers = []
    sizes = [n_feat, 256, 256, 128, 128, 64, 1]

    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i + 1]))
        if i < len(sizes) - 2:
            layers.append(nn.ReLU())
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(sizes[i + 1]))

    model = nn.Sequential(*layers).to(device)

    for m in model.modules():
        if isinstance(m, nn.Linear):
            if init_type == 'zeros':
                nn.init.zeros_(m.weight)
            elif init_type == 'large':
                nn.init.normal_(m.weight, std = 5.0)
            elif init_type == 'xavier':
                nn.init.xavier_normal_(m.weight)
            elif init_type == 'he':
                nn.init.kaiming_normal_(m.weight, nonlinearity = 'relu')

            nn.init.zeros_(m.bias)

    return model

# 3. Training
def train_model(model, n_epochs=30, desc=""):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    history = {'train': [], 'val': []}

    for epoch in range(n_epochs):
        model.train()
        tl, tn = 0.0, 0
        for bX, by in train_loader:
            bX, by = bX.to(device), by.to(device)
            optimizer.zero_grad()
            out  = model(bX)
            loss = criterion(out, by)

            if torch.isnan(loss):
                print(f"  [{desc}] NaN loss at epoch {epoch+1}! Stopping.")
                return history

            loss.backward()
            optimizer.step()
            tl += loss.item() * bX.size(0)
            tn += bX.size(0)

        model.eval()
        vl, vn = 0.0, 0
        with torch.no_grad():
            for bX, by in val_loader:
                bX, by = bX.to(device), by.to(device)
                vl += criterion(model(bX), by).item() * bX.size(0)
                vn += bX.size(0)

        history['train'].append(tl / tn)
        history['val'].append(vl / vn)

    return history

# 4. Experiment
experiments = [
    ("Zeros init, no BN", dict(use_batchnorm = False, init_type = 'zeros')),
    ("Large init, no BN", dict(use_batchnorm = False, init_type = 'large')),
    ("Xavier init, no BN", dict(use_batchnorm = False, init_type = 'xavier')),
    ("He init, no BN", dict(use_batchnorm = False, init_type = 'he')),
    ("He init + BatchNorm", dict(use_batchnorm = True,  init_type = 'he'))
]
print(f"\n{'Experiment':<30} {'Best Val Loss':>14} {'Final Val Loss':>16}")

all_histories = {}
for desc, cfg in experiments:
    model = build_model(**cfg)
    hist  = train_model(model, n_epochs=40, desc=desc)
    all_histories[desc] = hist

    if hist['val']:
        best  = min(hist['val'])
        final = hist['val'][-1]
        print(f"{desc:<30} {best:>14.4f} {final:>16.4f}")
    else:
        print(f"{desc:<30} {'FAILED (NaN)':>14}")

print("\n── Epochs to reach val loss < 0.60 ──")
threshold = 0.60
for desc, hist in all_histories.items():
    reached = next((i+1 for i, v in enumerate(hist['val']) if v < threshold), None)
    print(f"  {desc:<30}: {reached if reached else 'Never'}")


Experiment                      Best Val Loss   Final Val Loss
Zeros init, no BN                      0.6925           0.6925
Large init, no BN              432580159.1467   432597762.5600
Xavier init, no BN                     0.2622           0.4957
He init, no BN                         0.2652           0.4040
He init + BatchNorm                    0.4126           0.5245

── Epochs to reach val loss < 0.60 ──
  Zeros init, no BN             : Never
  Large init, no BN             : Never
  Xavier init, no BN            : 1
  He init, no BN                : 1
  He init + BatchNorm           : 1


In [13]:
# Mini Exercise
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def measure_activation_variance(init_std = None, use_he = False, use_xavier = False, 
                                use_bn = False, n_layers = 8, n_units = 256, n_in = 100):
    torch.manual_seed(42)
    x = torch.randn(128, n_in)

    stds = []
    for i in range(n_layers):
        in_f = n_in if i == 0 else n_units
        W = torch.randn(in_f, n_units)

        if use_he:
            W = W * (2.0 / in_f) ** 0.5
        elif use_xavier:
            W = W * (2.0 / (in_f + n_units)) ** 0.5
        elif init_std:
            W = W * init_std
        else:
            W = W * (1.0 / in_f) ** 0.5

        b = torch.zeros(n_units)
        x = torch.relu(x @ W + b)

        if use_bn:
            mean = x.mean(dim = 0, keepdim = True)
            var = x.var(dim = 0, keepdim = True, unbiased = False)
            x = (x - mean) / (var + 1e-5).sqrt()

        stds.append(x.std().item())
    
    return stds

print('Layer-wise activation std (8 deep ReLU network):')
print(f"\n{'Layer':>6} | {'Too Small':>10} | {'Too Large':>10} | {'He':>10} | {'He+BN':>10}")

stds_small = measure_activation_variance(init_std = 0.01)
stds_large = measure_activation_variance(init_std = 2.0)
stds_he = measure_activation_variance(use_he = True)
stds_he_bn = measure_activation_variance(use_he = True, use_bn = True)

for i in range(8):
    print(f"{i+1:>6} | {stds_small[i]:>10.4f} | {stds_large[i]:>10.2f} | "
          f"{stds_he[i]:>10.4f} | {stds_he_bn[i]:>10.4f}")

print('\n\nTraining Speed Comparison:')

n, f = 1000, 50
X = torch.randn(n, f);  y = (X[:, :3].sum(1) > 0).float().unsqueeze(1)
Xv = torch.randn(300, f); yv = (Xv[:, :3].sum(1) > 0).float().unsqueeze(1)

train_dl = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)
val_dl   = DataLoader(TensorDataset(Xv, yv), batch_size=128)

def make_deep_net(use_bn=False, init='he', n_layers=6, width=128):
    sizes = [f] + [width] * n_layers + [1]
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i+1]))
        if i < len(sizes) - 2:
            layers.append(nn.ReLU())
            if use_bn:
                layers.append(nn.BatchNorm1d(sizes[i+1]))

    model = nn.Sequential(*layers).to(device)

    for m in model.modules():
        if isinstance(m, nn.Linear):
            if init == 'he':
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif init == 'xavier':
                nn.init.xavier_normal_(m.weight)
            elif init == 'small':
                nn.init.normal_(m.weight, std=0.01)
            nn.init.zeros_(m.bias)
    return model

def quick_train(model, n_epochs=40):
    crit = nn.BCEWithLogitsLoss()
    opt  = optim.Adam(model.parameters(), lr=0.001)
    val_losses = []

    for ep in range(n_epochs):
        model.train()
        for bX, by in train_dl:
            bX, by = bX.to(device), by.to(device)
            opt.zero_grad()
            loss = crit(model(bX), by)
            if torch.isnan(loss): return val_losses, ep + 1
            loss.backward()
            opt.step()

        model.eval()
        vl = 0
        with torch.no_grad():
            for bX, by in val_dl:
                bX, by = bX.to(device), by.to(device)
                vl += crit(model(bX), by).item() * bX.size(0)
        val_losses.append(vl / len(val_dl.dataset))

    return val_losses, n_epochs

configs = [
    ("Small init, no BN",   dict(use_bn=False, init='small')),
    ("He init, no BN",      dict(use_bn=False, init='he')),
    ("Xavier init, no BN",  dict(use_bn=False, init='xavier')),
    ("He init + BatchNorm", dict(use_bn=True,  init='he')),
]

print(f"\n{'Config':<25} {'Best Val':>9} {'Epoch@Best':>11} {'Epoch<0.60':>12}")
print("-" * 62)

for desc, cfg in configs:
    model = make_deep_net(**cfg, n_layers=6, width=256)
    losses, ran = quick_train(model, n_epochs=40)

    if losses:
        best    = min(losses)
        best_ep = losses.index(best) + 1
        conv    = next((i+1 for i, v in enumerate(losses) if v < 0.60), None)
        print(f"{desc:<25} {best:>9.4f} {best_ep:>11} {conv if conv else 'Never':>12}")
    else:
        print(f"{desc:<25} {'NaN':>9}")

Layer-wise activation std (8 deep ReLU network):

 Layer |  Too Small |  Too Large |         He |      He+BN
     1 |     0.0592 |      11.83 |     0.8365 |     1.0000
     2 |     0.0068 |     271.07 |     0.8471 |     1.0000
     3 |     0.0007 |    5908.71 |     0.8160 |     1.0000
     4 |     0.0001 |  142783.75 |     0.8715 |     1.0000
     5 |     0.0000 | 2828548.50 |     0.7630 |     1.0000
     6 |     0.0000 | 64392356.00 |     0.7676 |     1.0000
     7 |     0.0000 | 1487800960.00 |     0.7838 |     1.0000
     8 |     0.0000 | 31463983104.00 |     0.7326 |     1.0000


Training Speed Comparison:

Config                     Best Val  Epoch@Best   Epoch<0.60
--------------------------------------------------------------
Small init, no BN            0.2444           8            2
He init, no BN               0.2905           3            1
Xavier init, no BN           0.2270           2            1
He init + BatchNorm          0.4580           4            1
